# Image Feature Extraction

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
dataset_name = 'MicroLens-100k'
os.chdir('./datasets/%s/5-core'%dataset_name)
os.getcwd()

## Load meta_data

In [ ]:
mapping_csv = os.path.join('./', 'i_id_mapping.csv')

df = pd.read_csv(mapping_csv, dtype=str, sep='\t')
df['itemID'] = df['itemID'].astype(int)
print("Columns:", df.columns.tolist())  # ['asin', 'itemID']
print(df.shape)
df[:5]

# Image Feature Exatrct

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch

gpu_id = 1
device = torch.device('cuda:' + str(gpu_id) if torch.cuda.is_available() else 'cpu')

model_name = "./pretrain_models/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name)
model = model.to(device)

def extract_features(image_path, model, processor):
    """ Use CLIP extract image feature """
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    inputs = inputs.to(model.device)
    with torch.no_grad():
        image_features =  model.get_image_features(**inputs)
    return image_features

image_folder = "../MicroLens-100k_covers"
asin_list = df['asin'].astype(str).tolist()
itemid_list = df['itemID'].tolist()

features = []
item_ids = [] # all items that have image
batch_size = 128
image_files = [f for f in os.listdir(image_folder) if f.endswith(".jpg")]
for i in tqdm(range(0, len(image_files), batch_size), desc="Processing Batches"):
    batch_asins = asin_list[i:i + batch_size]
    batch_itemids = itemid_list[i:i + batch_size]

    for asin, item_id in zip(batch_asins, batch_itemids):
        image_path = os.path.join(image_folder, f"{asin}.jpg")
        if not os.path.exists(image_path):
            print(f"[skip] Picture missing: {asin}.jpg")
            continue
        image_feature = extract_features(image_path, model, processor)

        image_feature = image_feature.detach().cpu().numpy()
        if len(features) == 0:
            features = image_feature
        else:
            features = np.concatenate((features, image_feature), axis=0)
        item_ids.append(item_id)

print(features.shape)
features[:5]

In [ ]:
average_feature = np.mean(features, axis=0)

# check null
features_array = np.empty((df.shape[0], features.shape[-1]))
idx = 0
for item_id in df['itemID'].values:
    if item_id not in item_ids:
        features_array[item_id] = average_feature 
    else:
        features_array[item_id] = features[idx] 
        idx += 1

assert features_array.shape[0] == df.shape[0]

print(features_array.shape)
features_array[:5]

In [ ]:
# save
np.save(os.path.join('./', 'image_feat.npy'), features_array)
print('done!')

In [ ]:
# load
load_img_feat = np.load('image_feat.npy', allow_pickle=True)
print(load_img_feat.shape)
load_img_feat[:10]